In [74]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

In [75]:
df=pd.read_csv(r'C:\Users\M S I\Desktop\5thsem_OJT\project1_diabetes\data\processed\hiba_p1w2_features.csv')
feature_cols=['age','pregnancies','glucose','blood_pressure','skin_thickness','insulin','bmi','diabetes_pedigree','bmi_category','glucose_category']

x=df[feature_cols].copy()
y=df['outcome']
x
x=pd.get_dummies(x,drop_first=True)
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)


In [76]:
def score_row(name, pred, score=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp)
    }
    if score is not None:
        r["auc"] = round(roc_auc_score(y_test, score), 4)
    return r

print("Train:", x_train.shape ,"Test:", x_test.shape)
print("DIABETIC PATIENTS IN THE TEST SET:", int(np.sum(y_test)), "of", len(y_test))
for nm, m, xt in [("Logistic Regression", logreg, x_test_scaled),
                  ("Decision Tree", dtree, x_test),
                  ("KNN", knn, x_test_scaled)]:     
    print(f"{nm:22s}  accuracy {accuracy_score(y_test,m.predict(xt)):.4f}")



Train: (760, 13) Test: (190, 13)
DIABETIC PATIENTS IN THE TEST SET: 55 of 190
Logistic Regression     accuracy 0.7526
Decision Tree           accuracy 0.7211
KNN                     accuracy 0.7474


## RESTATE THE TARGET YOU A

In [77]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182

print("PROJECT METRIC : recall  (guard: F1>=", GUARD_F1,")")
print("TARGET : recall >=", TARGET_RECALL)
print("BEST_SO_FAR :", BEST_SO_FAR, "(WEEK 5 BASELINE AT THRESHOLD 0.30)")
print("GAP TO CLOSE :", round(TARGET_RECALL - BEST_SO_FAR, 4))


PROJECT METRIC : recall  (guard: F1>= 0.5 )
TARGET : recall >= 0.7
BEST_SO_FAR : 0.6182 (WEEK 5 BASELINE AT THRESHOLD 0.30)
GAP TO CLOSE : 0.0818


## EXPLAIN HOW SVM REACHES A DECISION 

Support Vector Machine (SVM) is a supervised learning algorithm mainly used for classification.

SVM finds the best decision boundary (hyperplane) that separates different classes.

It tries to create the maximum margin between the classes.

Margin is the distance between the decision boundary and the closest data points from each class.

The closest data points are called Support Vectors.

The decision point is determined by the position of a new data point relative to the decision boundary.

If the point is on one side → Class 0.

If the point is on the other side → Class 1.

SVM uses a decision function to calculate this position.

A positive decision score usually indicates one class, while a negative score indicates the other class.

The decision boundary is where the decision score is 0.

SVM can also use kernel functions to separate data that cannot be separated by a straight line.

In [78]:
from sklearn.svm import SVC

In [79]:
svm_rbf = SVC(random_state=42)
svm_rbf.fit(x_train_scaled,y_train)
pred_rbf = svm_rbf.predict(x_test_scaled)
print("fitted:", svm_rbf)
print("patients flagged:",int(pred_rbf.sum()), "of", len(pred_rbf))

fitted: SVC(random_state=42)
patients flagged: 17 of 190


In [81]:
row_rbf = score_row("SVC rbc (default)", pred_rbf,
                    svm_rbf.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf]).to_string(index=False))
print()
print("target recall:", TARGET_RECALL, "| best so far:", BEST_SO_FAR)
print("this model  :", row_rbf["recall"])

            model  accuracy  precision  recall     f1  found  missed  false_alarms    auc
SVC rbc (default)    0.7263     0.5882  0.1818 0.2778     10      45             7 0.6443

target recall: 0.7 | best so far: 0.6182
this model  : 0.1818
